In [12]:
# ----------------------------------------------------------
# 1. Imports and cache setup
# ----------------------------------------------------------
import os
import fastf1
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

print("imports okay")

os.makedirs('fastf1_cache', exist_ok=True)
fastf1.Cache.enable_cache('fastf1_cache')

# ----------------------------------------------------------
# 2. Session selection
# ----------------------------------------------------------
year = 2023
event = 'Bahrain'
session_type = 'R'      # R = Race, Q = Qualifying, FP1, etc.
'''
# Driver codes – change these only
driver_a = 'VER'       # e.g. Hamilton
driver_b = 'HAM'        # e.g. Leclerc

# ----------------------------------------------------------
# 3. Load session and laps
# ----------------------------------------------------------
session = fastf1.get_session(year, event, session_type)
session.load()

laps = session.laps
print(f"Drivers in {event} {year}:", sorted(laps['Driver'].unique()))

# ----------------------------------------------------------
# 4. Helper to tidy one driver's laps
# ----------------------------------------------------------
def tidy_driver_laps(laps_df, driver_code, quick=True):
    """Return a tidy dataframe for one driver with key columns."""
    d = laps_df.pick_drivers(driver_code).copy()
    if quick:
        d = d.pick_quicklaps()  # remove pit in/out & outliers
    out = pd.DataFrame({
        'LapNumber' : d['LapNumber'],
        'LapStartTime' : d['LapStartTime'],
        'LapTime_s' : d['LapTime'].dt.total_seconds(),
        'Compound'  : d['Compound'],
        'TyreLife'  : d['TyreLife'],
        'Stint'     : d['Stint'],
        'FreshTyre' : d['FreshTyre']
    })
    out['Driver'] = driver_code
    return out.sort_values('LapNumber').reset_index(drop=True)

# ----------------------------------------------------------
# 5. Get both drivers' tidy tables
# ----------------------------------------------------------
a_tbl = tidy_driver_laps(laps, driver_a, quick=True)
b_tbl = tidy_driver_laps(laps, driver_b, quick=True)

print(f"{driver_a} laps:", len(a_tbl))
print(f"{driver_b} laps:", len(b_tbl))

#positions lap-by-lap
def positions_by_lap(laps_df, code):
    df = laps_df.pick_drivers(code)[['LapNumber','Position']].sort_values('LapNumber').copy()
    df['Position'] = df['Position'].ffill()
    df['Driver'] = code
    return df

pa = positions_by_lap(laps, driver_a)
pb = positions_by_lap(laps, driver_b)

plt.figure(figsize=(9,4))
plt.step(pa['LapNumber'], pa['Position'], where='post', label=driver_a)
plt.step(pb['LapNumber'], pb['Position'], where='post', label=driver_b)
plt.gca().invert_yaxis()
plt.xlabel('Lap number'); plt.ylabel('Race position')
plt.title(f'{event} {year} – Position by lap: {driver_a} vs {driver_b}')
plt.legend(); plt.tight_layout(); plt.show()
# ----------------------------------------------------------
# 6. Compare by lap number
# ----------------------------------------------------------
cmp = (a_tbl[['LapNumber','LapTime_s','Compound','TyreLife','Stint','FreshTyre']]
       .merge(
           b_tbl[['LapNumber','LapTime_s','Compound','TyreLife','Stint','FreshTyre']],
           on='LapNumber', how='outer',
           suffixes=(f'_{driver_a}', f'_{driver_b}'))
       .sort_values('LapNumber')
       .reset_index(drop=True))

cmp[f'Delta_{driver_a}_minus_{driver_b}_s'] = (
    cmp[f'LapTime_s_{driver_a}'] - cmp[f'LapTime_s_{driver_b}']
)

# Preview
print(cmp.head(10))

# ----------------------------------------------------------
# 7. Plot lap-time traces
# ----------------------------------------------------------
plt.figure(figsize=(8,5))
plt.plot(cmp['LapNumber'], cmp[f'LapTime_s_{driver_a}'],
         marker='o', label=driver_a)
plt.plot(cmp['LapNumber'], cmp[f'LapTime_s_{driver_b}'],
         marker='o', label=driver_b)
plt.xlabel('Lap number'); plt.ylabel('Lap time (s)')
plt.title(f'{event} {year} – {driver_a} vs {driver_b} Lap Times')
plt.legend(); plt.tight_layout(); plt.show()

# ----------------------------------------------------------
# 8. Plot lap-by-lap delta
# ----------------------------------------------------------
plt.figure(figsize=(8,4))
plt.plot(cmp['LapNumber'], cmp[f'Delta_{driver_a}_minus_{driver_b}_s'],
         marker='o', color='purple')
plt.axhline(0, linestyle='--', color='grey')
plt.xlabel('Lap number'); plt.ylabel(f'Delta {driver_a} − {driver_b} (s)')
plt.title(f'Lap-by-lap Delta – {driver_a} faster if below 0')
plt.tight_layout(); plt.show()

# ----------------------------------------------------------
# 9. (Optional) Average lap time by tyre compound
# ----------------------------------------------------------
summary = (pd.concat([a_tbl, b_tbl])
           .groupby(['Driver','Compound'])['LapTime_s']
           .mean()
           .unstack(0)
           .round(3))
print("\nAverage lap time by compound (s):")
print(summary) '''

# ----------------------------------------------------------
# 1. Function to build lap-level dataset for one race
# ----------------------------------------------------------
def make_lap_dataset_for_session(year, event, session_type='R'):
    """Creates a DataFrame with one row per lap per driver for a given race."""
    print(f"Loading session: {event} {year} {session_type}...")
    session = fastf1.get_session(year, event, session_type)
    session.load()
    print(f"  -> Session loaded, laps: {len(session.laps)}")

    laps = session.laps.copy()
    laps = laps[~laps['Driver'].isna()].copy()

    # ---- per-driver features ----
    def add_driver_features(df):
        df = df.sort_values('LapNumber').copy()
        laps_since_pit = []
        counter = 0
        for _, row in df.iterrows():
            counter += 1
            laps_since_pit.append(counter)
            if pd.notna(row['PitInTime']):
                counter = 0
        df['LapsSinceLastPit'] = laps_since_pit
        df['StopsSoFar'] = df['Stint'] - 1
        return df

    laps = laps.groupby('Driver', group_keys=False).apply(add_driver_features)

    # ---- target 1: pit on THIS lap ----
    laps['is_pit_lap'] = laps['PitInTime'].notna().astype(int)

    # ---- target 2: pit on the NEXT lap ----
    laps = laps.sort_values(['Driver', 'LapNumber']).copy()
    laps['will_pit_next_lap'] = (
        laps.groupby('Driver')['is_pit_lap']
            .shift(-1)          # move the pit flag one lap back
            .fillna(0)
            .astype(int)
    )

    # Optional: drop the actual pit laps themselves if you only
    # want to predict from "normal" laps
    laps = laps[laps['is_pit_lap'] == 0].copy()

    # ---- other features ----
    total_laps = laps['LapNumber'].max()
    laps['LapNumberNorm'] = laps['LapNumber'] / total_laps
    laps['TrackStatus'] = laps['TrackStatus'].astype(str)

    feature_cols = [
        'Driver', 'Compound', 'Stint', 'LapsSinceLastPit',
        'StopsSoFar', 'LapNumber', 'LapNumberNorm',
        'Position', 'TrackStatus'
    ]

    target_cols = ['is_pit_lap', 'will_pit_next_lap']

    data = laps[feature_cols + target_cols].dropna()
    data['Track'] = event
    data['Year'] = year

    print(f"  -> Data rows after cleaning: {len(data)}")
    print("  -> will_pit_next_lap counts:", data['will_pit_next_lap'].value_counts().to_dict())
    return data


# ----------------------------------------------------------
# 2. Build training and test datasets from different races
# ----------------------------------------------------------

# Races used for TRAINING
print("Building training races...")
train_races = [
    (2023, 'Bahrain'),
    (2023, 'Saudi Arabia'),
    (2023, 'Australia'),
    (2023, 'China'),
    (2023, 'Azerbaijan'),
    (2023, 'Miami'),
    (2023, 'Emilia Romagna'),
    (2023, 'Monaco'),
    (2023, 'Spain'),
    (2023, 'Canada')
]

# Race used for TESTING (unseen race)
test_race = (2023, 'Austria')

train_dfs = []
for year, event in train_races:
    df = make_lap_dataset_for_session(year, event, 'R')
    train_dfs.append(df)

train_dataset = pd.concat(train_dfs, ignore_index=True)

print("\nTRAIN columns:", train_dataset.columns.tolist())
print("TRAIN will_pit_next_lap counts:", train_dataset['will_pit_next_lap'].value_counts())

print(f"\nBuilding TEST race: {test_race[1]} {test_race[0]}")
test_dataset = make_lap_dataset_for_session(test_race[0], test_race[1], 'R')
print("TEST will_pit_next_lap counts:", test_dataset['will_pit_next_lap'].value_counts())

# ----------------------------------------------------------
# 3. Features and preprocessing
# ----------------------------------------------------------
feature_cols = [
    'Driver', 'Compound', 'Stint', 'LapsSinceLastPit',
    'StopsSoFar', 'LapNumber', 'LapNumberNorm',
    'Position', 'TrackStatus'
]

target_col = 'will_pit_next_lap'

X_train = train_dataset[feature_cols]
y_train = train_dataset[target_col]

X_test = test_dataset[feature_cols]
y_test = test_dataset[target_col]

categorical_features = ['Driver', 'Compound', 'TrackStatus']
numeric_features = [
    'Stint', 'LapsSinceLastPit', 'StopsSoFar',
    'LapNumber', 'LapNumberNorm', 'Position'
]

preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ('num', 'passthrough', numeric_features)
])

clf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight='balanced'   # still needed: pit laps are rare
)

model = Pipeline([
    ('preprocess', preprocess),
    ('clf', clf)
])

# ----------------------------------------------------------
# 4. Train on TRAIN races
# ----------------------------------------------------------
print("\nTraining model on training races...")
model.fit(X_train, y_train)

# ----------------------------------------------------------
# 5. Evaluate on COMPLETELY DIFFERENT race
# ----------------------------------------------------------
print(f"\nEvaluating on unseen race: {test_race[1]} {test_race[0]}...")
y_pred = model.predict(X_test)

print("\nClassification report:\n", classification_report(y_test, y_pred))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))

# ----------------------------------------------------------
# 6. Feature importance (still across all training data)
# ----------------------------------------------------------
ohe = model.named_steps['preprocess'].named_transformers_['cat']
cat_names = ohe.get_feature_names_out(categorical_features)
all_features = list(cat_names) + numeric_features
rf = model.named_steps['clf']
importances = rf.feature_importances_

feat_imp = pd.DataFrame({
    'Feature': all_features,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("\nTop 10 most important features (trained on multiple races):\n",
      feat_imp.head(10))




imports okay
Building training races...
Loading session: Bahrain 2023 R...


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.6.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '14', '55', '44', '18', '63', '77', '10', '23', '22', '2', '20', '21', '27', '24', '4', '31', '16', '81']


  -> Session loaded, laps: 1056


C:\Users\Josel\AppData\Local\Temp\ipykernel_23412\2000720066.py:167: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  laps = laps.groupby('Driver', group_keys=False).apply(add_driver_features)
core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.6.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


  -> Data rows after cleaning: 1003
  -> will_pit_next_lap counts: {0: 951, 1: 52}
Loading session: Saudi Arabia 2023 R...


...

req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '11', '4', '14', '55', '63', '44', '18', '10', '23', '24', '2', '31', '77', '81', '21', '20', '22', '27']


  -> Session loaded, laps: 1354


C:\Users\Josel\AppData\Local\Temp\ipykernel_23412\2000720066.py:167: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  laps = laps.groupby('Driver', group_keys=False).apply(add_driver_features)


  -> Data rows after cleaning: 1290
  -> will_pit_next_lap counts: {0: 1231, 1: 59}
TEST will_pit_next_lap counts: will_pit_next_lap
0    1231
1      59
Name: count, dtype: int64

Training model on training races...

Evaluating on unseen race: Austria 2023...

Classification report:
               precision    recall  f1-score   support

           0       0.95      1.00      0.98      1231
           1       0.00      0.00      0.00        59

    accuracy                           0.95      1290
   macro avg       0.48      0.50      0.49      1290
weighted avg       0.91      0.95      0.93      1290


Confusion matrix:
 [[1231    0]
 [  59    0]]

Top 10 most important features (trained on multiple races):
              Feature  Importance
44     LapNumberNorm    0.213700
41  LapsSinceLastPit    0.169422
43         LapNumber    0.157236
45          Position    0.101069
40             Stint    0.041369
42        StopsSoFar    0.039957
21     Compound_HARD    0.031152
27     TrackSta

C:\Users\Josel\anaconda3\envs\f1ai\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Josel\anaconda3\envs\f1ai\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Josel\anaconda3\envs\f1ai\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0]